In [ ]:
import tensorflow as tf
import numpy as np
import random

np.random.seed(17)
tf.random.set_seed(17)
random.seed(17)

# ограничим потребление видеопамяти
# GPUs = tf.config.list_physical_devices('GPU')
# tf.config.experimental.set_memory_growth(GPUs[0], True)

In [ ]:
densenet = tf.keras.applications.DenseNet201()

82524592/82524592 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step


In [ ]:
# изучим её архитектуру

densenet.summary()

Model: "densenet201"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 20,242,984 (77.22 MB)

 Trainable params: 20,013,928 (76.35 MB)

 Non-trainable params: 229,056 (894.75 KB)

In [ ]:
# для корректной работы с картинками загрузим модуль pillow

!pip install pillow

In [ ]:
cat_img = tf.keras.preprocessing.image.load_img('cat.jpg', target_size=(224, 224))

In [ ]:
cat_img.show()

In [ ]:
# далее необходимо представить картинку в виде массива, где матрицами закодирована интенсивность пикселей по нашим разным каналам

cat_img = tf.keras.preprocessing.image.img_to_array(cat_img)

In [ ]:
# 3 канала и 224х224 пикселя
cat_img.shape

(224, 224, 3)

Т.к на вход нейросети нужно подавать массив картинок, то сначала сделаем массив из одной картинки, а затем специально подготовим его для ResNet функцией preprocess_input, которая отмасштабирует значения пикселей на нужную величину

In [ ]:
cat_img = np.expand_dims(cat_img, axis=0) # нейронные сети ожидают батч изображений
cat_img = tf.keras.applications.densenet.preprocess_input(cat_img) # resnet обучен imagenet с определенной предобработкой

In [ ]:
cat_img

array([[[[1.0330508, 1.7633053, 2.5528543],
         [0.9988013, 1.7282913, 2.5179958],
         [0.9988013, 1.7282913, 2.5179958],
         ...,
         [1.2042983, 1.8333333, 2.5702832],
         [1.2042983, 1.8333333, 2.5702832],
         [1.2042983, 1.8333333, 2.5702832]],

        [[1.015926 , 1.7457983, 2.535425 ],
         [0.9988013, 1.7282913, 2.5179958],
         [0.9988013, 1.7282913, 2.5179958],
         ...,
         [1.2042983, 1.8333333, 2.5702832],
         [1.2042983, 1.8333333, 2.5702832],
         [1.2042983, 1.8333333, 2.5702832]],

        [[1.015926 , 1.7457983, 2.535425 ],
         [0.9988013, 1.7282913, 2.5179958],
         [0.9988013, 1.7282913, 2.5179958],
         ...,
         [1.2042983, 1.8333333, 2.5702832],
         [1.2042983, 1.8333333, 2.5702832],
         [1.2042983, 1.8333333, 2.5702832]],

        ...,

        [[1.6837914, 2.2009804, 2.6051416],
         [1.7180408, 2.2009804, 2.622571 ],
         [1.7180408, 2.2009804, 2.622571 ],
         ...,


In [ ]:
# получаем предсказания

pred = densenet.predict(cat_img)

1/1 ━━━━━━━━━━━━━━━━━━━━ 8s 8s/step


In [ ]:
# получили список из 1000 вероятностей
pred

array([[3.27646325e-04, 3.51719573e-05, 1.07397800e-05, 3.39734979e-04,
        2.27026394e-05, 1.88989594e-04, 1.84108887e-06, 4.69302364e-07,
        8.31897978e-06, 7.36175707e-06, 1.86921079e-05, 3.72935892e-06,
        1.21619569e-05, 1.45710117e-04, 1.67801245e-05, 2.45420906e-05,
        4.74119634e-06, 3.14612917e-05, 7.70966755e-04, 1.76476385e-06,
        3.86389176e-04, 7.65261484e-06, 3.65764527e-06, 1.75458743e-04,
        2.75166309e-03, 9.78293428e-06, 9.10122253e-05, 1.84485980e-05,
        2.52868398e-04, 6.76690252e-04, 3.94380113e-05, 8.78454921e-06,
        3.97220902e-05, 3.39058170e-06, 4.24383899e-07, 4.97798246e-05,
        1.50268737e-04, 2.77873914e-06, 3.39354992e-05, 1.48817426e-05,
        5.59918908e-06, 1.35164628e-05, 2.38488292e-06, 4.76431487e-05,
        8.69780342e-06, 1.25020722e-06, 5.24523921e-06, 8.81019296e-05,
        4.45683763e-05, 3.84031046e-06, 1.44653222e-06, 2.57548436e-05,
        1.36687631e-05, 1.74369725e-05, 4.41670227e-06, 6.642403

In [ ]:
# чтобы расшифровать это дело, есть специальная функция

tf.keras.applications.densenet.decode_predictions(pred)

35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


[[('n02124075', 'Egyptian_cat', np.float32(0.16923495)),
  ('n01882714', 'koala', np.float32(0.14802775)),
  ('n01883070', 'wombat', np.float32(0.11369907)),
  ('n02108915', 'French_bulldog', np.float32(0.10923619)),
  ('n02125311', 'cougar', np.float32(0.045628052))]]

In [ ]:
# заморозим все слои

for layer in densenet.layers:
  layer.trainable = False

# но несколько последних разморозим обратно
# число 20 здесь взято для примера, а вообще конечно
# это тоже гиперпараметр, который нужно подбирать
# оптимальным может оказаться любое значение от 1 до переобучения половины сети

for layer in densenet.layers[-10:]:
  layer.trainable = True

# заменим активацию на последнем слое

densenet.layers[-1].activation = tf.keras.activations.relu

In [ ]:
densenet.summary()

Model: "densenet201"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d      │ (None, 230, 230,  │          0 │ input_layer[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_conv (Conv2D) │ (None, 112, 112,  │      9,408 │ zero_padding2d[0… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_bn            │ (None, 112, 112,  │        256 │ conv1_conv[0][0]  │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1_relu          │ (None, 112, 112,  │          0 │ conv1_bn[0][0]    │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ zero_padding2d_1    │ (None, 114, 114,  │          0 │ conv1_relu[0][0]  │
│ (ZeroPadding2D)     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool1               │ (None, 56, 56,    │          0 │ zero_padding2d_1… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_bn   │ (None, 56, 56,    │        256 │ pool1[0][0]       │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_0_relu │ (None, 56, 56,    │          0 │ conv2_block1_0_b… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_conv │ (None, 56, 56,    │      8,192 │ conv2_block1_0_r… │
│ (Conv2D)            │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_bn   │ (None, 56, 56,    │        512 │ conv2_block1_1_c… │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_1_relu │ (None, 56, 56,    │          0 │ conv2_block1_1_b… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_2_conv │ (None, 56, 56,    │     36,864 │ conv2_block1_1_r… │
│ (Conv2D)            │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block1_concat │ (None, 56, 56,    │          0 │ pool1[0][0],      │
│ (Concatenate)       │ 96)               │            │ conv2_block1_2_c… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_bn   │ (None, 56, 56,    │        384 │ conv2_block1_con… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_0_relu │ (None, 56, 56,    │          0 │ conv2_block2_0_b… │
│ (Activation)        │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2_block2_1_conv │ (None, 56, 56,    │     12,288 │ conv2_block2_0_r

 Total params: 20,242,984 (77.22 MB)

 Trainable params: 2,203,624 (8.41 MB)

 Non-trainable params: 18,039,360 (68.81 MB)

Теперь сделаем новую модель на базе старой.Просто включим весь наш resnet в качестве слоя и добавим к нему все необходимое

In [ ]:
model_cats = tf.keras.models.Sequential([
    densenet, # вся модель выступает в качестве слоя
    #tf.keras.layers.BatchNormalization(),
    #tf.keras.layers.Dropout(0.5), # по-хорошему нужно подбирать
    tf.keras.layers.Dense(1, activation='sigmoid') # слой для бинарной классификации
])

In [ ]:
model_cats.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ densenet201 (Functional)        │ (None, 1000)           │    20,242,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │         1,001 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,243,985 (77.22 MB)

 Trainable params: 2,204,625 (8.41 MB)

 Non-trainable params: 18,039,360 (68.81 MB)

In [ ]:
# скомпилируем получившуюся модель, добавив необходимые метрики

accuracy = tf.keras.metrics.binary_accuracy
precision = tf.keras.metrics.Precision()
recall = tf.keras.metrics.Recall()

def f1_metrics(y_true, y_pred):
  prec = precision(y_true, y_pred)
  rec = recall(y_true, y_pred)
  return 2 * ((prec * rec) / (prec + rec + 1e-7))

model_cats.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.00001),
                   loss=tf.keras.losses.binary_crossentropy,
                   metrics=[accuracy, precision, recall, f1_metrics])

Сформируем датасет для тренировки модели. Перегоним картинки в массив.

In [ ]:
# функция предподготовки картинки для модели

def preprocess_image(file):
  img = tf.keras.preprocessing.image.load_img(file, target_size=(224, 224)) # загружаем в нужном разрешении
  img = tf.keras.preprocessing.image.img_to_array(img) # конвертируем в массив
  img = tf.keras.applications.densenet.preprocess_input(img) # препроцессинг для densenet
  return img

Пробежимся по всем файлам в наших папках и добавим их в соответсвующие списки.К каждой картинке добавим лейбл: если кот- 1, в иных случаях- 0. Это поможет не запутаться в данных при перемешивании.

In [ ]:
import os

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving neural_networks_block-main.zip to neural_networks_block-main.zip


In [ ]:
import zipfile

with zipfile.ZipFile('neural_networks_block-main.zip', 'r') as zip_ref:
  zip_ref.extractall('./')

In [ ]:
# добавляем пары (картинка, 1) для картинок с котами
cats = [(preprocess_image('neural_networks_block-main/5. Свёрточные нейронные сети/pics/cats/'+file), 1) for file in os.listdir('neural_networks_block-main/5. Свёрточные нейронные сети/pics/cats')]

# и пары (картинка, 0) для картинок без котов
nocats = [(preprocess_image('neural_networks_block-main/5. Свёрточные нейронные сети/pics/nocats/'+file), 0) for file in os.listdir('neural_networks_block-main/5. Свёрточные нейронные сети/pics/nocats')]

In [ ]:
# сливаем оба списка вместе

all_pics = cats + nocats

In [ ]:
# перемешиваем данные

random.shuffle(all_pics)

In [ ]:
# в x отправляем картинки, а в y - прикрепленные к ним лейблы

x = np.array([a[0] for a in all_pics])
y = np.array([a[1] for a in all_pics])

In [ ]:
# делим данные на трейн, валидацию и тест

def train_val_test_split(x, val_frac=0.15, test_frac=0.15):
  x_train = x[:round((1 - val_frac - test_frac) * len(x))]
  x_val = x[round((1 - val_frac - test_frac) * len(x)):round((1 - test_frac) * len(x))]
  x_test = x[round((1 - test_frac) * len(x)):]
  return x_train, x_val, x_test

x_train, x_val, x_test = train_val_test_split(x)
y_train, y_val, y_test = train_val_test_split(y)

In [ ]:
# настроек у этого класса куда больше, но для примера возьмем только самые основные

datagen = tf.keras.preprocessing.image.ImageDataGenerator(rotation_range=45, # случайный поворот в пределах 45 градусов
                                                          width_shift_range=0.2, # случайный сдвиг по горизонтали
                                                          height_shift_range=0.2, # и вертикали
                                                          horizontal_flip=True, # случайное отражение по горизнотали
                                                          vertical_flip=True)

# также у ImageDataGenerator есть полезный аргумент preprocessing_function,
# в котором можно указать любую свою функцию для отработки изображения

datagen.fit(x_train)


In [ ]:
# будем отслеживать обучение в tensorboard

tb_callback = tf.keras.callbacks.TensorBoard(log_dir='logs/tl_resnet_cats', histogram_freg=1)

# и уменьшать lr на плато

annealing = tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=10, verbose=1)

In [ ]:
bs = 16

# вместо самих данных подаём в цикл обучения картинки из нашего генератора
model_cats.fit(datagen.flow(x_train, y_train, batch_size=bs),
               validation_data=(x_val, y_val),
               steps_per_epoch=len(x_train)//bs, # чтобы генератор не уходил в бесконечный цикл, указываем кол-во ошибок
               epochs=50)
               #callbacks=[tb_callback, annealing])

Epoch 1/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 127s 5s/step - binary_accuracy: 0.5922 - f1_metrics: 0.2247 - loss: 0.7290 - precision_1: 0.4151 - recall_1: 0.2054 - val_binary_accuracy: 0.5000 - val_f1_metrics: 0.3098 - val_loss: 1.8068 - val_precision_1: 0.3333 - val_recall_1: 0.3600
Epoch 2/50
 1/20 ━━━━━━━━━━━━━━━━━━━━ 1:05 3s/step - binary_accuracy: 0.8125 - f1_metrics: 0.5714 - loss: 0.4609 - precision_1: 1.0000 - recall_1: 0.4000

/usr/local/lib/python3.12/dist-packages/keras/src/trainers/epoch_iterator.py:116: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


20/20 ━━━━━━━━━━━━━━━━━━━━ 20s 868ms/step - binary_accuracy: 0.8125 - f1_metrics: 0.5714 - loss: 0.4609 - precision_1: 1.0000 - recall_1: 0.4000 - val_binary_accuracy: 0.5000 - val_f1_metrics: 0.3098 - val_loss: 1.7760 - val_precision_1: 0.3333 - val_recall_1: 0.3600
Epoch 3/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 94s 5s/step - binary_accuracy: 0.6280 - f1_metrics: 0.4337 - loss: 0.6792 - precision_1: 0.5632 - recall_1: 0.3649 - val_binary_accuracy: 0.5735 - val_f1_metrics: 0.3166 - val_loss: 1.4160 - val_precision_1: 0.4000 - val_recall_1: 0.3200
Epoch 4/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 19s 841ms/step - binary_accuracy: 0.6250 - f1_metrics: 0.5000 - loss: 0.7304 - precision_1: 0.6000 - recall_1: 0.4286 - val_binary_accuracy: 0.5735 - val_f1_metrics: 0.3166 - val_loss: 1.4027 - val_precision_1: 0.4000 - val_recall_1: 0.3200
Epoch 5/50
20/20 ━━━━━━━━━━━━━━━━━━━━ 121s 5s/step - binary_accuracy: 0.6768 - f1_metrics: 0.4341 - loss: 0.6278 - precision_1: 0.5587 - recall_1: 0.3976 - val_binary_accuracy: 

In [ ]:
# проверим модель на тестовых данных
model_cats.evaluate(x_test, y_test)

In [ ]:
model_cats.predict(cat_img)